# 💰 Multi-City Code Enforcement Financial Analysis
## Deadline: November 10, 2025

**Data Strategy**: Using legacy clean data now, designed for easy switch to schema-extracted data later.

**Cities Analyzed**:
-  **Margate**: 741 violations (complete dataset)
-  **Boca Raton**: Processed violations
-  **Pompano Beach**: Processed violations  
-  **Wilton Manor**: Processed violations

**Key Metrics**:
- Violation frequency by city
- Financial impact analysis
- Temporal trends
- Geographic patterns
- Risk assessment scores

In [1]:
# ANALYSIS CONFIGURATION
# Professional formatting preferences

import warnings
warnings.filterwarnings('ignore')

# Display settings
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_rows', 100)

# Path display settings - control path visibility
from pathlib import Path
import os

# Professional output preferences
PROFESSIONAL_MODE = True
SHOW_FULL_PATHS = False  # Set to True if you want to see full paths

def print_path(label, path):
    """Print path only if SHOW_FULL_PATHS is True"""
    if SHOW_FULL_PATHS:
        print(f"{label}: {path}")
    else:
        print(f"{label}: [hidden]")

print("Configuration loaded:")
print(f"- Professional mode: {PROFESSIONAL_MODE}")
print(f"- Show full paths: {SHOW_FULL_PATHS}")
print("- Display options optimized")
print("- Warnings suppressed")

Configuration loaded:
- Professional mode: True
- Show full paths: False
- Display options optimized
- Warnings suppressed


## Data Loading & Standardization

**Design**: Standardized data loading that works with both legacy and schema data

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set up paths
ROOT = Path.cwd().parent
CLEAN_DATA_DIR = ROOT / "clean_data"
SCHEMA_DATA_DIR = ROOT / "results_folder_schema"

# Use configuration settings for path display
print_path("Project root", ROOT)
print_path("Clean data", CLEAN_DATA_DIR)
print_path("Schema data", SCHEMA_DATA_DIR)

# Configure plots
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)

print("Libraries loaded and configured")

Project root: [hidden]
Clean data: [hidden]
Schema data: [hidden]
Libraries loaded and configured


In [3]:
def load_city_data(city_name: str, use_schema: bool = False) -> pd.DataFrame:
    """
    Load city data from either legacy clean files or schema-extracted files
    
    Args:
        city_name: Name of city (margate, boca_raton, etc.)
        use_schema: If True, try schema data first, fallback to legacy
        
    Returns:
        Standardized DataFrame with consistent column names
    """
    
    df = None
    data_source = "unknown"
    
    # Try schema data first if requested
    if use_schema:
        schema_files = list((SCHEMA_DATA_DIR / city_name).glob("*_clean.csv"))
        if schema_files:
            try:
                df = pd.read_csv(schema_files[0])
                data_source = "schema"
                print(f"📊 {city_name}: Loaded {len(df)} rows from schema data")
            except Exception as e:
                print(f"⚠️  Schema data failed for {city_name}: {e}")
    
    # Fallback to legacy data
    if df is None:
        legacy_patterns = [
            f"{city_name}_clean.csv",
            f"{city_name}_processed.csv", 
            f"{city_name.replace('_', ' ').title().replace(' ', '')}_clean.csv"
        ]
        
        for pattern in legacy_patterns:
            legacy_file = CLEAN_DATA_DIR / pattern
            if legacy_file.exists():
                try:
                    df = pd.read_csv(legacy_file)
                    data_source = "legacy"
                    print(f"📋 {city_name}: Loaded {len(df)} rows from legacy data")
                    break
                except Exception as e:
                    print(f"⚠️  Failed to load {legacy_file}: {e}")
    
    if df is None:
        print(f"❌ No data found for {city_name}")
        return pd.DataFrame()
    
    # Standardize column names for consistent analysis
    df = standardize_columns(df, data_source)
    df['city'] = city_name.replace('_', ' ').title()
    df['data_source'] = data_source
    
    return df

def standardize_columns(df: pd.DataFrame, source: str) -> pd.DataFrame:
    """
    Standardize column names for consistent analysis
    """
    
    column_mapping = {}
    
    if source == "schema":
        # Schema data has clean names already
        column_mapping = {
            'CASE NUMBER': 'violation_id',
            'CASE TYPE': 'violation_type', 
            'ADDRESS': 'address',
            'STATUS': 'status',
            'DATE OPENED': 'date_opened',
            'DAYS ACTIVE': 'days_active',
            'LAST ACTION': 'last_action',
            'NEXT ACTION': 'next_action',
            'RESULT DATE': 'result_date',
            'DUE DATE': 'due_date'
        }
    else:
        # Legacy data needs cleaning
        column_mapping = {
            'violation_id_raw': 'violation_id',
            'violation_description_raw': 'violation_type',
            'address_raw': 'address', 
            'case_status_raw': 'status',
            'opened_date_raw': 'date_opened',
            'days_active_raw': 'days_active',
            'last_action_raw': 'last_action',
            'next_action_raw': 'next_action',
            'result_date_raw': 'result_date',
            'due_date_raw': 'due_date',
            'closed_date_raw': 'date_closed'
        }
    
    # Apply mappings
    df_clean = df.copy()
    for old_col, new_col in column_mapping.items():
        if old_col in df_clean.columns:
            df_clean[new_col] = df_clean[old_col]
    
    return df_clean

print("✅ Data loading functions defined")
print("🔄 Designed for easy switching between legacy and schema data")

✅ Data loading functions defined
🔄 Designed for easy switching between legacy and schema data


## Load All City Data

Loading all available city data with automatic fallback from schema to legacy data.

In [4]:
# Load all city data
cities = ['margate', 'pompano_beach', 'wilton_manor'] ##only clean data cities
city_data = {}
total_violations = 0

print(" LOADING MULTI-CITY VIOLATION DATA")
print("=" * 50)

for city in cities:
    df = load_city_data(city, use_schema=False)  # Using legacy for deadline
    if not df.empty:
        city_data[city] = df
        total_violations += len(df)
        
        # Quick data quality check
        key_fields = ['violation_id', 'address', 'status']
        completeness = {}
        for field in key_fields:
            if field in df.columns:
                completeness[field] = df[field].notna().sum() / len(df) * 100
        
        print(f"   Data quality: {completeness}")

print(f"\n SUMMARY:")
print(f"   Cities loaded: {len(city_data)}")
print(f"   Total violations: {total_violations:,}")
print(f"   Data sources: {[df['data_source'].iloc[0] for df in city_data.values()]}")

if len(city_data) == 0:
    print("❌ No data loaded! Check file paths.")
else:
    print(f" Ready for financial analysis!")

 LOADING MULTI-CITY VIOLATION DATA
📋 margate: Loaded 741 rows from legacy data
   Data quality: {'violation_id': np.float64(100.0), 'address': np.float64(90.55330634278003), 'status': np.float64(100.0)}
📋 pompano_beach: Loaded 389 rows from legacy data
   Data quality: {'violation_id': np.float64(95.11568123393316), 'address': np.float64(97.17223650385604), 'status': np.float64(95.11568123393316)}
📋 wilton_manor: Loaded 568 rows from legacy data
   Data quality: {}

 SUMMARY:
   Cities loaded: 3
   Total violations: 1,698
   Data sources: ['legacy', 'legacy', 'legacy']
 Ready for financial analysis!


##  Financial Impact Analysis

Analyzing the financial implications of code enforcement violations across all cities.

In [5]:
# Combine all city data for analysis
if city_data:
    all_violations = pd.concat(city_data.values(), ignore_index=True)
    
    print(f" MULTI-CITY VIOLATION ANALYSIS")
    print(f" Total violations: {len(all_violations):,}")
    print(f" Cities: {all_violations['city'].unique()}")
    
    # Violations by city
    city_counts = all_violations['city'].value_counts()
    print(f"\n VIOLATIONS BY CITY:")
    for city, count in city_counts.items():
        percentage = count / len(all_violations) * 100
        print(f"   {city}: {count:,} violations ({percentage:.1f}%)")
    
    # Status distribution
    if 'status' in all_violations.columns:
        status_counts = all_violations['status'].value_counts()
        print(f"\n STATUS DISTRIBUTION:")
        for status, count in status_counts.head().items():
            percentage = count / len(all_violations) * 100
            print(f"   {status}: {count:,} ({percentage:.1f}%)")
            
    print(f"\n Basic analysis complete - ready for detailed financial modeling!")
else:
    print("❌ No data available for analysis")

 MULTI-CITY VIOLATION ANALYSIS
 Total violations: 1,698
 Cities: ['Margate' 'Pompano Beach' 'Wilton Manor']

 VIOLATIONS BY CITY:
   Margate: 741 violations (43.6%)
   Wilton Manor: 568 violations (33.5%)
   Pompano Beach: 389 violations (22.9%)

 STATUS DISTRIBUTION:
   ACTIVE: 864 (50.9%)
   CASE CLOSED: 239 (14.1%)
   IN COMPLIANCE/OPEN FINES: 7 (0.4%)
   VOIDED: 1 (0.1%)

 Basic analysis complete - ready for detailed financial modeling!


## Master Dataset Examination

Detailed view of the consolidated violation data structure and contents

In [9]:
# Examine the master consolidated dataset
if 'all_violations' in locals() and not all_violations.empty:
    
    print("MASTER DATASET OVERVIEW")
    print("=" * 50)
    print(f"Total rows: {len(all_violations):,}")
    print(f"Total columns: {len(all_violations.columns)}")
    print(f"Cities included: {list(all_violations['city'].unique())}")
    print(f"Date range: {all_violations.index.min()} to {all_violations.index.max()}")
    
    print(f"\nCOLUMN STRUCTURE:")
    print("-" * 30)
    for i, col in enumerate(all_violations.columns, 1):
        non_null_count = all_violations[col].notna().sum()
        completeness = (non_null_count / len(all_violations)) * 100
        print(f"{i:2d}. {col:<25} | {non_null_count:,} values ({completeness:.1f}% complete)")
    
    print(f"\nDATA TYPES:")
    print("-" * 20)
    print(all_violations.dtypes)
    
    print(f"\nFIRST 5 ROWS SAMPLE:")
    print("-" * 25)
    print(all_violations.head())
    
    print(f"\nMEMORY USAGE:")
    print("-" * 15)
    memory_usage = all_violations.memory_usage(deep=True).sum() / (1024 * 1024)  # Convert to MB
    print(f"Dataset size: {memory_usage:.2f} MB")
    
else:
    print("No master dataset found. Run the previous cell first.")

MASTER DATASET OVERVIEW
Total rows: 1,698
Total columns: 34
Cities included: ['Margate', 'Pompano Beach', 'Wilton Manor']
Date range: 0 to 1697

COLUMN STRUCTURE:
------------------------------
 1. violation_id_raw          | 1,111 values (65.4% complete)
 2. case_status_raw           | 1,111 values (65.4% complete)
 3. chunk_id                  | 1,698 values (100.0% complete)
 4. source_file               | 1,698 values (100.0% complete)
 5. Unnamed: 4                | 17 values (1.0% complete)
 6. violation_description_raw | 1,119 values (65.9% complete)
 7. address_raw               | 1,049 values (61.8% complete)
 8. opened_date_raw           | 1,111 values (65.4% complete)
 9. days_active_raw           | 1,111 values (65.4% complete)
10. last_action_raw           | 1,088 values (64.1% complete)
11. next_action_raw           | 702 values (41.3% complete)
12. closed_date_raw           | 224 values (13.2% complete)
13. result_date_raw           | 680 values (40.0% complete)
14. due_

## Export Master Dataset to CSV

Save the consolidated dataset for external examination and backup

In [8]:
# Export consolidated dataset to CSV file
if 'all_violations' in locals() and not all_violations.empty:
    
    # Create output filename with timestamp 
    from datetime import datetime
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_filename = f"master_violations_dataset_{timestamp}.csv"
    output_path = ROOT / "clean_data" / output_filename
    
    # Export to CSV
    all_violations.to_csv(output_path, index=False)
    
    print("CSV EXPORT COMPLETE")
    print("=" * 30)
    print(f"File: {output_filename}")
    print(f"Location: {output_path}")
    print(f"Records exported: {len(all_violations):,}")
    print(f"Columns exported: {len(all_violations.columns)}")
    print(f"File size: {output_path.stat().st_size / (1024*1024):.2f} MB")
    
    # Show what was exported
    print(f"\nEXPORTED COLUMNS:")
    print("-" * 20)
    for i, col in enumerate(all_violations.columns, 1):
        print(f"{i:2d}. {col}")
        
    print(f"\nCITIES INCLUDED:")
    print("-" * 20)
    city_counts = all_violations['city'].value_counts()
    for city, count in city_counts.items():
        print(f"{city}: {count:,} violations")
    
    print(f"\nYou can now open this CSV file in Excel, Google Sheets, or any spreadsheet application!")
    
else:
    print("No master dataset found. Run the data loading cells first.")

CSV EXPORT COMPLETE
File: master_violations_dataset_20251110_112857.csv
Location: c:\Users\edilm\OneDrive\Desktop\Edilma Projects\LandingAI-Hack\coderisk-sf\clean_data\master_violations_dataset_20251110_112857.csv
Records exported: 1,698
Columns exported: 34
File size: 0.43 MB

EXPORTED COLUMNS:
--------------------
 1. violation_id_raw
 2. case_status_raw
 3. chunk_id
 4. source_file
 5. Unnamed: 4
 6. violation_description_raw
 7. address_raw
 8. opened_date_raw
 9. days_active_raw
10. last_action_raw
11. next_action_raw
12. closed_date_raw
13. result_date_raw
14. due_date_raw
15. city
16. violation_id
17. violation_type
18. address
19. status
20. date_opened
21. days_active
22. last_action
23. next_action
24. result_date
25. due_date
26. date_closed
27. data_source
28. violation_code_raw
29. case_disposition_raw
30. 0
31. 1
32. 2
33. 3
34. 4

CITIES INCLUDED:
--------------------
Margate: 741 violations
Wilton Manor: 568 violations
Pompano Beach: 389 violations

You can now open thi

## Violation Type Analysis

Exploring unique violation types across all cities to identify investment opportunity patterns

In [10]:
# Analyze violation types across all cities
if 'all_violations' in locals() and not all_violations.empty:
    
    print("VIOLATION TYPE ANALYSIS")
    print("=" * 50)
    
    # Overall violation type statistics
    if 'violation_type' in all_violations.columns:
        violation_counts = all_violations['violation_type'].value_counts()
        
        print(f"Total unique violation types: {len(violation_counts)}")
        print(f"Most common violations across all cities:")
        print("-" * 40)
        
        # Show top 15 most common violations
        for i, (violation, count) in enumerate(violation_counts.head(15).items(), 1):
            percentage = (count / len(all_violations)) * 100
            print(f"{i:2d}. {violation:<50} | {count:4d} ({percentage:5.1f}%)")
        
        # Show violation types by city
        print(f"\nVIOLATION TYPES BY CITY:")
        print("=" * 40)
        
        for city in all_violations['city'].unique():
            city_data = all_violations[all_violations['city'] == city]
            city_violations = city_data['violation_type'].value_counts()
            
            print(f"\n{city.upper()}:")
            print(f"  Unique violation types: {len(city_violations)}")
            print(f"  Top 5 violations:")
            
            for i, (violation, count) in enumerate(city_violations.head(5).items(), 1):
                percentage = (count / len(city_data)) * 100
                print(f"    {i}. {violation:<40} | {count:4d} ({percentage:4.1f}%)")
        
        # Identify potential investment categories
        print(f"\nPOTENTIAL INVESTMENT CATEGORIES:")
        print("=" * 40)
        
        # Keywords that might indicate different investment opportunities
        building_keywords = ['building', 'structure', 'construction', 'permit', 'zoning']
        maintenance_keywords = ['maintenance', 'repair', 'property', 'yard', 'exterior']
        safety_keywords = ['safety', 'fire', 'electrical', 'plumbing', 'code']
        
        building_violations = []
        maintenance_violations = []
        safety_violations = []
        other_violations = []
        
        for violation in violation_counts.index:
            violation_lower = str(violation).lower()
            
            if any(keyword in violation_lower for keyword in building_keywords):
                building_violations.append((violation, violation_counts[violation]))
            elif any(keyword in violation_lower for keyword in maintenance_keywords):
                maintenance_violations.append((violation, violation_counts[violation]))
            elif any(keyword in violation_lower for keyword in safety_keywords):
                safety_violations.append((violation, violation_counts[violation]))
            else:
                other_violations.append((violation, violation_counts[violation]))
        
        print(f"BUILDING/DEVELOPMENT Related: {len(building_violations)} types")
        if building_violations:
            for violation, count in sorted(building_violations, key=lambda x: x[1], reverse=True)[:5]:
                print(f"  - {violation} ({count})")
        
        print(f"\nMAINTENANCE/PROPERTY Related: {len(maintenance_violations)} types")
        if maintenance_violations:
            for violation, count in sorted(maintenance_violations, key=lambda x: x[1], reverse=True)[:5]:
                print(f"  - {violation} ({count})")
        
        print(f"\nSAFETY/CODE Related: {len(safety_violations)} types")
        if safety_violations:
            for violation, count in sorted(safety_violations, key=lambda x: x[1], reverse=True)[:5]:
                print(f"  - {violation} ({count})")
        
        print(f"\nOTHER: {len(other_violations)} types")
        if other_violations:
            for violation, count in sorted(other_violations, key=lambda x: x[1], reverse=True)[:5]:
                print(f"  - {violation} ({count})")
        
        print(f"\nREADY FOR INVESTMENT CATEGORIZATION!")
        print("Review the violation types above to decide investment opportunity categories.")
        
    else:
        print("No violation_type column found in dataset")
        print("Available columns:", list(all_violations.columns))
        
else:
    print("No master dataset found. Run the data loading cells first.")

VIOLATION TYPE ANALYSIS
Total unique violation types: 169
Most common violations across all cities:
----------------------------------------
 1. WORK WITHOUT PERMIT-BUILDING                       |  389 ( 22.9%)
 2. 40 YEAR SAFETY INSPECTIONS                         |  155 (  9.1%)
 3. UNSAFE STRUCTURE (BLDG)                            |  100 (  5.9%)
 4. SPECIFIC VIOL; OCCUPY STUCTURE                     |   22 (  1.3%)
 5. BTR; RENTAL HOUSING                                |   21 (  1.2%)
 6. SPECIFIC VIOL; DISTUB LANDSCAP                     |   21 (  1.2%)
 7. BTR; RENTAL HOUSING INSPECTION                     |   18 (  1.1%)
 8. 50 YEAR SAFETY INSPECTIONS                         |   18 (  1.1%)
 9. PARKING; SURFACE STANDARDS                         |   17 (  1.0%)
10. NUISANCE; GRASS OR WEEDS                           |   13 (  0.8%)
11. ABANDONED VEHICLES                                 |   13 (  0.8%)
12. NUISANCE; STATE OF REPAIR                          |   13 (  0.8%)
13. REN